# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MasoomSakina/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Archetype to Action Mapping and Queue Ranking

To turn model probability scores and impression signals into operational workflows, content items are mapped to four primary action archetypes:

* **High Impression Decay (`RECOMMEND_REFRESH`):** Content items with $\ge 1,000$ 90-day impressions and a downward trend trajectory (`trend_direction == 'down'`). High priority for editorial update.
* **Low Exposure Decay (`MONITOR`):** Content items with $< 1,000$ impressions and a downward trend trajectory. Low priority; kept in monitoring queue to avoid wasting resources on low-impact pages.
* **Stable / Growing Content (`NO_ACTION`):** Content items experiencing neutral or upward trend trajectories (`trend_direction` in `['up', 'stable']`). Kept untouched to preserve existing search equity.
* **New / Low Exposure Content (`NO_ACTION`):** Newly published or unindexed items without sufficient baseline impression volume.

Items in the action queue are ranked by priority score and impression volume, providing editorial leads with an actionable, transparent decision-support tool.

In [7]:
import pandas as pd
import numpy as np

# Load local CSV dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Define playbook action assignment logic
def assign_action_playbook(row):
    impressions = row.get('impressions_90d', 0)
    trend = row.get('trend_direction', 'flat')
    
    if impressions >= 1000 and trend == 'down':
        return pd.Series(['RECOMMEND_REFRESH', 'HIGH_IMPRESSION_DECAY', 0.85])
    elif impressions < 1000 and trend == 'down':
        return pd.Series(['MONITOR', 'LOW_IMPRESSION_DECAY', 0.40])
    elif trend in ['up', 'stable']:
        return pd.Series(['NO_ACTION', 'STABLE_GROWING_TRAFFIC', 0.10])
    else:
        return pd.Series(['NO_ACTION', 'NEUTRAL_TRAFFIC_PROFILE', 0.10])

df[['action', 'reason_code', 'priority_score']] = df.apply(assign_action_playbook, axis=1)
ranked_queue = df.sort_values(by=['priority_score', 'impressions_90d'], ascending=[False, False])

print("--- TOP 5 RANKED RECOMMENDATIONS IN QUEUE ---")
display(ranked_queue[['impressions_90d', 'trend_direction', 'action', 'reason_code', 'priority_score']].head())

--- TOP 5 RANKED RECOMMENDATIONS IN QUEUE ---


,impressions_90d,trend_direction,action,reason_code,priority_score
6653,517715,down,RECOMMEND_REFRESH,HIGH_IMPRESSION_DECAY,0.85
26844,509252,down,RECOMMEND_REFRESH,HIGH_IMPRESSION_DECAY,0.85
21819,463103,down,RECOMMEND_REFRESH,HIGH_IMPRESSION_DECAY,0.85
29879,416180,down,RECOMMEND_REFRESH,HIGH_IMPRESSION_DECAY,0.85
13537,347399,down,RECOMMEND_REFRESH,HIGH_IMPRESSION_DECAY,0.85


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use and Operational Limits

* **Intended Use:** Serves as a decision-support queue for editorial leads, SEO teams, and content strategists to prioritize editorial refresh cycles on high-exposure pages suffering traffic decay.
* **Operational Limits:**
  * **No Causal Proof:** The queue identifies *where* traffic decay is occurring but does not diagnose *why* (e.g., search engine algorithm updates vs. technical site errors vs. shifting intent).
  * **No Ranking Guarantees:** Flagging a page for refresh does not guarantee a recovery in search engine impression share.
  * **Domain Scope:** Valid strictly for indexed, published content with established impression histories.

In [8]:
# Summary verification of queue bounds and action breakdowns
action_counts = ranked_queue['action'].value_counts()
print("--- PLAYBOOK QUEUE ACTION BREAKDOWN ---")
print(action_counts)

--- PLAYBOOK QUEUE ACTION BREAKDOWN ---
action
NO_ACTION            13738
MONITOR               8231
RECOMMEND_REFRESH     8031
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Protocol & The No-Go Automation List

**Human Review Checklist (Mandatory before taking action):**
1. **Migration & URL Audit:** Verify whether recent traffic changes reflect structural URL moves, 301 redirects, or site migrations rather than content staleness.
2. **Search Intent Alignment:** Confirm whether target search query intent has fundamentally shifted before making editorial adjustments.
3. **Seasonality Check:** Verify that downward trend vectors are not caused by regular annual seasonality.

**The No-Go List (STRICTLY PROHIBITED FROM AUTOMATION):**
* **Automated Rewriting & Publishing:** Never auto-generate and push live content updates without human editorial sign-off.
* **Automated Page Deletions or Redirects:** Never execute automated 301 redirects or page deletions based solely on baseline model flags.
* **Automated Canonical or Schema Adjustments:** Structural site changes must remain under manual engineering control.

In [9]:
# Identify potential human-review boundary cases (e.g., items near 1,000 impression cutoff)
borderline_cases = ranked_queue[(ranked_queue['impressions_90d'] >= 950) & (ranked_queue['impressions_90d'] <= 1050)]
print(f"Borderline items requiring manual review before action: {len(borderline_cases):,}")

Borderline items requiring manual review before action: 476


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring Signals & Model Retrain Triggers

To prevent recommendations from going stale or degrading over time, the playbook relies on explicit monitoring triggers:

* **Data Drift Triggers:** Re-evaluate thresholds if the proportion of decaying pages (`trend_direction == 'down'`) deviates by more than $\pm 10\%$ between 30-day evaluation windows.
* **Performance Decay Triggers:** If audited refresh actions fail to show directional recovery over a subsequent 90-day window, re-audit validation split assumptions.
* **Scheduled Retraining:** Retrain models quarterly or immediately following major platform indexing changes and large-scale client migrations.

In [10]:
# Calculate baseline drift metric for distribution monitoring
decay_proportion = (ranked_queue['trend_direction'] == 'down').mean()
print(f"Current Baseline Decay Rate: {decay_proportion:.2%}")
print(f"Monitoring Drift Alert Thresholds: < {decay_proportion - 0.10:.2%} or > {decay_proportion + 0.10:.2%}")

Current Baseline Decay Rate: 54.21%
Monitoring Drift Alert Thresholds: < 44.21% or > 64.21%


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for Research Paper Integration

This section exports the generated queue to `work/outputs/` and exports summary visualizations to `work/figures/` for direct reference in the final research paper report.

In [11]:
import os
import matplotlib.pyplot as plt

# 1. Export Ranked Queue to work/outputs/
os.makedirs('../../work/outputs', exist_ok=True)
output_queue_path = '../../work/outputs/content_action_playbook_queue.csv'
ranked_queue.to_csv(output_queue_path, index=False)
print(f"Successfully exported ranked queue to: {output_queue_path}")

# 2. Export Figure to work/figures/ using Matplotlib
os.makedirs('../../work/figures', exist_ok=True)
fig_path = '../../work/figures/action_distribution.png'

action_counts = ranked_queue['action'].value_counts()

plt.figure(figsize=(8, 4))
plt.bar(action_counts.index, action_counts.values, color='#2b5c8f')
plt.title('Distribution of Content Actions in Playbook Queue')
plt.xlabel('Recommended Action')
plt.ylabel('Content Item Count')
plt.tight_layout()
plt.savefig(fig_path, dpi=300)
plt.close()

print(f"Successfully exported summary figure to: {fig_path}")

Successfully exported ranked queue to: ../../work/outputs/content_action_playbook_queue.csv
Successfully exported summary figure to: ../../work/figures/action_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it[cite: 10]
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)[cite: 10]
- [x] No client names, URLs, or private queries anywhere[cite: 10]
- [x] My claims use careful words: observed, measured, directional, decision-support[cite: 10]
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.[cite: 10]